# Trích xuất đồ thị mã nguồn cho tập dữ liệu DiverseVul

Quy trình trích xuất hai biểu diễn đồ thị cho 9.077 hàm:
- Graph A: đồ thị cây cú pháp trừu tượng AST
- Graph B: đồ thị ngữ nghĩa kết hợp AST, luồng điều khiển CFG và phụ thuộc dữ liệu DATA_DEP

## 1. Môi trường

Internet để On. Accelerator chọn CPU. Dataset input là diversevul-raw.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "feature/3_3_graph-extract"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/kaggle/working") / "software-bug-detection-using-graphs"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(["git", *args], cwd=cwd, check=True)

if not REPO_DIR.exists():
    git(["clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=str(REPO_DIR))
    git(["checkout", GIT_BRANCH], cwd=str(REPO_DIR))
    git(["pull", "origin", GIT_BRANCH], cwd=str(REPO_DIR))

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
!pip install -q pyyaml

## 2. Cài đặt OpenJDK 21 và Joern

Cài đặt máy ảo Java và công cụ phân tích tĩnh Joern. Phiên bản Joern được ghim bằng `JOERN_VERSION` vì installer không có tham số `--version` sẽ tải bản mới nhất.

In [ ]:
JOERN_VERSION = "v4.0.654"

# Cài đặt OpenJDK 21
!apt-get update -qq && apt-get install -y -qq openjdk-21-jdk

# Cài đặt Joern đúng phiên bản đã ghim
!curl -sSL https://github.com/joernio/joern/releases/download/{JOERN_VERSION}/joern-install.sh | bash -s -- --version={JOERN_VERSION}

# Thiết lập biến môi trường
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = "/root/bin:" + os.environ["PATH"]

# Kiểm tra phiên bản công cụ
import shutil

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=20)
print(java_check.stderr)

# Không gọi `joern --version`: Joern 4.x không có cờ này, lệnh mở REPL chờ stdin và làm cell treo.
for tool in ("joern", "joern-parse"):
    tool_path = shutil.which(tool)
    print(tool, "->", tool_path)
    assert tool_path, f"Không tìm thấy {tool} sau khi cài đặt"

# Ghi nhận phiên bản đã cài từ tên tệp thư viện, không mở REPL.
print("Phiên bản yêu cầu:", JOERN_VERSION)
!ls /opt/joern/joern-cli/lib 2>/dev/null | grep -i joern | head -5

## 3. Xác thực dữ liệu đầu vào

Kiểm tra tệp dữ liệu mã nguồn DiverseVul và manifest thí nghiệm 9.077 hàm.

In [ ]:
dataset_candidates = [
    Path("/kaggle/input/diversevul-raw/diversevul.jsonl"),
    Path("/kaggle/input/diversevul-dataset/diversevul.jsonl"),
    Path("/kaggle/input/diversevul/diversevul.jsonl"),
]
for p in Path("/kaggle/input").rglob("diversevul.jsonl"):
    if p.is_file() and p not in dataset_candidates:
        dataset_candidates.insert(0, p)

dataset_path = next((p for p in dataset_candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Không tìm thấy tệp diversevul.jsonl trong /kaggle/input. Cần thêm dataset qua nút Add Input.")

manifest_path = Path("data/splits/diversevul_experiment_manifest.csv")
if not manifest_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy manifest tại {manifest_path}")

with open(manifest_path, "r", encoding="utf-8") as f:
    sample_count = sum(1 for _ in f) - 1

print("Đường dẫn tệp mã nguồn:", dataset_path)
print("Dung lượng tệp:", round(dataset_path.stat().st_size / 1048576, 2), "MB")
print("Đường dẫn manifest:", manifest_path)
print("Tổng số mẫu thí nghiệm:", sample_count)

## 4. Thực thi trích xuất đồ thị theo phân đoạn

Graph A là đồ thị cây cú pháp trừu tượng AST
Graph B là đồ thị ngữ nghĩa kết hợp AST, luồng điều khiển CFG và phụ thuộc dữ liệu DATA_DEP

9.077 mẫu được chia thành 5 phân đoạn liên tiếp (khoảng 1.815 mẫu mỗi phân đoạn), mỗi phân đoạn chạy trong một version Kaggle riêng. Đổi giá trị `SESSION` từ 1 đến 5 rồi chọn Save & Run All. Các version chạy độc lập nên có thể khởi chạy lần lượt mà không cần chờ version trước kết thúc.

Tham số `--max-runtime-hours 11` làm tiến trình ngừng nhận mẫu mới sau 11 giờ và thoát bình thường để version lưu được kết quả trước giới hạn 12 giờ của Kaggle. Nếu phân đoạn chưa xong, thêm output của version đó vào Input rồi chạy lại cùng `SESSION`, các mẫu đã xong sẽ được bỏ qua.

In [ ]:
# =========================================================================
# CHỌN PHÂN ĐOẠN: ĐỔI SESSION TỪ 1 ĐẾN 5 TRƯỚC MỖI LẦN SAVE & RUN ALL
# 1: mẫu 0-1816 | 2: 1816-3632 | 3: 3632-5447 | 4: 5447-7262 | 5: 7262-9077
# =========================================================================
SESSION = 1

# Thư mục chứa kết quả của lần chạy trước (Add Input từ output của version cũ).
# Mẫu đã có đủ Graph A và Graph B sẽ được bỏ qua. Không có kết quả cũ thì không ảnh hưởng.
RESUME_FROM = "/kaggle/input"

!python scripts/extract_subset.py --dataset {dataset_path} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_session_{SESSION} --batch {SESSION} --workers 4 --max-runtime-hours 11 --resume-from {RESUME_FROM}

## 5. Kiểm tra và đóng gói tạo tác kết quả

Kiểm tra số lượng đồ thị đã trích xuất và đóng gói kết quả dạng tệp nén zip để lưu trữ.

In [ ]:
import glob

from scripts.extract_subset import BATCH_DEFINITIONS

session_start, session_end = BATCH_DEFINITIONS[SESSION]
expected_count = session_end - session_start
print("Phân đoạn", SESSION, "- số mẫu mong đợi:", expected_count)

output_directories = sorted(glob.glob("/kaggle/working/output_session_*"))
for out_dir in output_directories:
    p_dir = Path(out_dir)
    count_a = len(list((p_dir / "graph_a").glob("*.json"))) if (p_dir / "graph_a").is_dir() else 0
    count_b = len(list((p_dir / "graph_b").glob("*.json"))) if (p_dir / "graph_b").is_dir() else 0
    print("Thư mục đầu ra:", p_dir.name)
    print("Số lượng đồ thị cây cú pháp trừu tượng AST Graph A:", count_a)
    print("Số lượng đồ thị ngữ nghĩa kết hợp Graph B:", count_b)
    if count_a != expected_count or count_b != expected_count:
        print("Cảnh báo: phân đoạn chưa đủ mẫu, thiếu", expected_count - min(count_a, count_b),
              "mẫu. Thêm output của version này vào Input rồi chạy lại cùng SESSION.")

    batch_name = p_dir.name
    target_archive = f"/kaggle/working/{batch_name}.zip"
    print("Thực hiện đóng gói thư mục:", out_dir)
    !zip -q -r {target_archive} {out_dir}
    archive_size = os.path.getsize(target_archive) / 1048576
    print("Tệp lưu trữ hoàn tất:", target_archive)
    print("Dung lượng tệp:", round(archive_size, 2), "MB")